# AviSense — ML risk classifier (Google Colab)

The main AviSense risk score is a transparent weighted formula (`configs/risk.yaml`).
This notebook trains an **optional ML classifier** on human-labelled clips and
compares it with the formula. It also **learns formula weights from your labels**,
so the ML can be used to calibrate the explainable score rather than replace it.

**Why Colab?** scikit-learn is blocked on the development laptop by Windows
Application Control. Colab has it preinstalled. The exported model is plain JSON
and runs locally with NumPy only (`src/risk_scoring/ml_model.py`).

## What you need

For each labelled video, two CSV files with the same prefix:

1. **`<video>_risk.csv`** — per-frame output of the batch pipeline, run locally:
   ```
   python -m src.tracking.track --source data/videos/<video>.mp4 --csv data/processed/<video>_tracks.csv --imgsz 1280
   python -m src.risk_scoring.features --tracks data/processed/<video>_tracks.csv --zones configs/zones/<cam>.json --out data/processed/<video>_features.csv
   python -m src.risk_scoring.risk --features data/processed/<video>_features.csv --out data/processed/<video>_risk.csv
   ```
2. **`<video>_labels.csv`** — your judgement of the risk while watching the clip,
   as time ranges (a spreadsheet is fine). Frames outside every range are ignored.
   ```
   start_s,end_s,label
   0,12.5,Low
   12.5,20,Medium
   20,31,High
   ```

Label **at least 3–5 videos** with a mix of all three levels; splits are by video so
the model is always tested on footage it has not seen.

> **Demo mode** (`DEMO_MODE = True`) uses the formula's own levels as labels, only to
> check the notebook runs end to end. Results in demo mode say nothing about real
> accuracy — the classifier is just copying the formula.

In [ ]:
DEMO_MODE = False          # True = use formula levels as labels (pipeline check only)
DATA_DIR = "avisense_data" # folder with *_risk.csv and *_labels.csv

import glob, json, os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

LEVELS = ["Low", "Medium", "High"]
TERMS = ["n_count", "n_speed", "n_proximity", "n_flock_size", "n_time_of_day"]  # the formula's inputs
LOGREG_FEATURES = TERMS + ["n_in_zone", "n_approaching", "mean_speed"]
RF_FEATURES = LOGREG_FEATURES + ["count", "flock_size", "n_flocks", "max_speed"]
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Load data
Either upload the CSVs (next cell) or set `DATA_DIR` to a Google Drive folder.

In [ ]:
try:
    from google.colab import drive, files
    USE_DRIVE = False  # set True and edit DATA_DIR to read from Drive instead
    if USE_DRIVE:
        drive.mount("/content/drive")
        DATA_DIR = "/content/drive/MyDrive/avisense_data"
    else:
        for name, content in files.upload().items():
            Path(DATA_DIR, name).write_bytes(content)
except ImportError:
    print("Not running in Colab; reading from", DATA_DIR)

print(sorted(os.listdir(DATA_DIR)))

In [ ]:
def attach_labels(risk: pd.DataFrame, labels: pd.DataFrame) -> pd.Series:
    out = pd.Series(pd.NA, index=risk.index, dtype="object")
    for seg in labels.itertuples():
        if seg.label not in LEVELS:
            raise ValueError(f"Unknown label {seg.label!r}; use {LEVELS}")
        out[(risk["time_s"] >= seg.start_s) & (risk["time_s"] < seg.end_s)] = seg.label
    return out

frames = []
for risk_path in sorted(glob.glob(f"{DATA_DIR}/*_risk.csv")):
    video = Path(risk_path).name.removesuffix("_risk.csv")
    risk = pd.read_csv(risk_path)
    if DEMO_MODE:
        risk["label"] = risk["risk_level"]
    else:
        label_path = Path(DATA_DIR, f"{video}_labels.csv")
        if not label_path.exists():
            print(f"skip {video}: no {label_path.name}")
            continue
        risk["label"] = attach_labels(risk, pd.read_csv(label_path))
    risk["video"] = video
    frames.append(risk.dropna(subset=["label"]))

data = pd.concat(frames, ignore_index=True)
data["y"] = data["label"].map(LEVELS.index)
print(f"{len(data)} labelled frames from {data['video'].nunique()} video(s)")
print(data.groupby(["video", "label"]).size().unstack(fill_value=0).reindex(columns=LEVELS, fill_value=0))
if DEMO_MODE:
    print("\nDEMO MODE: labels are the formula's own output. Do not report these scores.")

## 2. Cross-validation setup
Frames from the same video are highly correlated, so folds are split **by video**.
With a single video we fall back to 5 contiguous time blocks (weaker, but no
frame-level leakage).

In [ ]:
n_videos = data["video"].nunique()
if n_videos >= 2:
    groups = data["video"]
    cv = GroupKFold(n_splits=min(5, n_videos))
else:
    groups = pd.qcut(data["time_s"].rank(method="first"), 5, labels=False)
    cv = GroupKFold(n_splits=5)
    print("Only one video: using 5 contiguous time blocks as folds.")

def cross_val_predict_levels(fit_predict):
    pred = np.empty(len(data), dtype=int)
    for tr, te in cv.split(data, data["y"], groups):
        pred[te] = fit_predict(data.iloc[tr], data.iloc[te])
    return pred

def report(name, pred):
    print(f"=== {name}: macro-F1 {f1_score(data['y'], pred, average='macro', labels=[0, 1, 2]):.3f}")
    print(classification_report(data["y"], pred, labels=[0, 1, 2], target_names=LEVELS, zero_division=0))
    print(pd.DataFrame(confusion_matrix(data["y"], pred, labels=[0, 1, 2]),
                       index=[f"true {l}" for l in LEVELS], columns=[f"pred {l}" for l in LEVELS]))

results = {}

## 3. Baseline: the current formula (`configs/risk.yaml`)

In [ ]:
formula_pred = data["risk_level"].map(LEVELS.index).to_numpy()
report("Formula (current weights)", formula_pred)
results["Formula (current)"] = f1_score(data["y"], formula_pred, average="macro", labels=[0, 1, 2])

## 4. Logistic regression (exportable, runs locally with NumPy)

In [ ]:
def make_logreg():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(class_weight="balanced", max_iter=2000))

def logreg_fit_predict(tr, te):
    m = make_logreg().fit(tr[LOGREG_FEATURES], tr["y"])
    return m.predict(te[LOGREG_FEATURES])

pred = cross_val_predict_levels(logreg_fit_predict)
report("Logistic regression", pred)
results["Logistic regression"] = f1_score(data["y"], pred, average="macro", labels=[0, 1, 2])

## 5. Random forest (stronger baseline, not exported)

In [ ]:
def rf_fit_predict(tr, te):
    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced",
                               random_state=0, n_jobs=-1)
    return m.fit(tr[RF_FEATURES], tr["y"]).predict(te[RF_FEATURES])

pred = cross_val_predict_levels(rf_fit_predict)
report("Random forest", pred)
results["Random forest"] = f1_score(data["y"], pred, average="macro", labels=[0, 1, 2])

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced",
                            random_state=0, n_jobs=-1).fit(data[RF_FEATURES], data["y"])
print(pd.Series(rf.feature_importances_, index=RF_FEATURES).sort_values(ascending=False).round(3))

## 6. Learn formula weights from your labels

Fit non-negative weights for the five formula terms so that the weighted sum tracks
the labelled level (Low=0, Medium=0.5, High=1), then pick the Medium/High
thresholds that maximise macro-F1. The result is still the same transparent
formula — just calibrated on data.

In [ ]:
def fit_weights(df):
    lr = LinearRegression(positive=True, fit_intercept=False).fit(df[TERMS], df["y"] / 2)
    w = lr.coef_ if lr.coef_.sum() > 0 else np.ones(len(TERMS))
    return w / w.sum()

def best_thresholds(score, y):
    best = (0.3, 0.6, -1.0)
    for med in np.arange(0.05, 0.95, 0.025):
        for high in np.arange(med + 0.025, 1.0, 0.025):
            pred = np.where(score >= high, 2, np.where(score >= med, 1, 0))
            f1 = f1_score(y, pred, average="macro", labels=[0, 1, 2])
            if f1 > best[2]:
                best = (med, high, f1)
    return best

def learned_formula_fit_predict(tr, te):
    w = fit_weights(tr)
    med, high, _ = best_thresholds(tr[TERMS].to_numpy() @ w, tr["y"])
    s = te[TERMS].to_numpy() @ w
    return np.where(s >= high, 2, np.where(s >= med, 1, 0))

pred = cross_val_predict_levels(learned_formula_fit_predict)
report("Formula (learned weights)", pred)
results["Formula (learned)"] = f1_score(data["y"], pred, average="macro", labels=[0, 1, 2])

w = fit_weights(data)
med, high, _ = best_thresholds(data[TERMS].to_numpy() @ w, data["y"])
names = [t.removeprefix("n_") for t in TERMS]
print("\nSuggested configs/risk.yaml values (fit on all labelled data):\n")
print("weights:")
for n, v in zip(names, w):
    print(f"  {n}: {v:.2f}")
print(f"levels:\n  medium: {med:.2f}\n  high: {high:.2f}")

## 7. Comparison

In [ ]:
summary = pd.Series(results, name="macro-F1 (cross-validated)").sort_values(ascending=False).round(3)
print(summary.to_string())
if DEMO_MODE:
    print("\nDEMO MODE: not a real evaluation.")

## 8. Export the logistic regression for local use

Download `risk_model.json`, put it in `models/`, and run locally:
```
python -m src.risk_scoring.risk --features data/processed/<video>_features.csv --out data/processed/<video>_risk.csv --ml-model models/risk_model.json
```

In [ ]:
final = make_logreg().fit(data[LOGREG_FEATURES], data["y"])
scaler, clf = final.named_steps["standardscaler"], final.named_steps["logisticregression"]
coef, intercept = clf.coef_, clf.intercept_
if len(clf.classes_) == 2:  # binary case: sklearn stores one row; expand to two classes
    coef, intercept = np.vstack([-coef[0] / 2, coef[0] / 2]), np.array([-intercept[0] / 2, intercept[0] / 2])

spec = {
    "type": "logreg",
    "classes": [LEVELS[c] for c in clf.classes_],
    "features": LOGREG_FEATURES,
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist(),
    "coef": coef.tolist(),
    "intercept": intercept.tolist(),
    "trained_on": {"videos": sorted(data["video"].unique()), "frames": int(len(data)),
                   "demo_mode": DEMO_MODE, "cv_macro_f1": round(results["Logistic regression"], 3)},
}
Path("risk_model.json").write_text(json.dumps(spec, indent=2))

# Sanity check: the NumPy re-implementation used locally must match sklearn.
x = (data[LOGREG_FEATURES].to_numpy() - np.array(spec["mean"])) / np.array(spec["scale"])
logits = x @ np.array(spec["coef"]).T + np.array(spec["intercept"])
assert (clf.classes_[logits.argmax(axis=1)] == final.predict(data[LOGREG_FEATURES])).all()
print("Exported risk_model.json; NumPy predictions match sklearn.")

try:
    from google.colab import files
    files.download("risk_model.json")
except ImportError:
    pass